# Training Naive Bayes' model
When training the Decision Tree model on the full training set, we kept running into memory errors. Thus, we decided to cut down on our dataset and use a different model that is also suitable on multiclass label prediction. 

One of the biggest issues with our low accuracy in previous models was the imbalanced subreddit. To solve this, we will filter for the top 100 subreddits, which should all have >5 entries.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, HashingTF, StringIndexer
from pyspark.ml.classification import NaiveBayes
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

spark = SparkSession.builder \
    .appName("Top100-Subreddit-Classifier") \
    .config("spark.driver.memory", "16g") \
    .config("spark.executor.memory", "16g") \
    .config("spark.driver.memoryOverhead", "4g") \
    .config("spark.executor.memoryOverhead", "4g") \
    .config("spark.sql.files.ignoreCorruptFiles", "true") \
    .getOrCreate()

In [ ]:
# LOAD & CLEAN DATA
df = spark.read.parquet('data/filtered2')
# dropped some columns to avoid OOM error (common error)
df = df.drop('over_18')
df = df.drop('post_id')
df = df.drop('link_flair_text')
df = df.drop('title')
TARGET_COL = "subreddit" 

df_clean = df.dropna(subset=["self_text", TARGET_COL])
df_clean = df_clean.filter(df_clean.self_text != "")
df_clean = df_clean.filter(F.length(F.col("self_text")) < 50000)

print("Finding the top 100 most popular subreddits...")
top_100_df = df_clean.groupBy(TARGET_COL) \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(100) \
    .select(TARGET_COL)

df_top_100 = df_clean.join(top_100_df, on=TARGET_COL, how="inner")

# THE LINEAGE BREAKER 
print("Writing intermediate data to disk to free up memory...")
# We write the joined data to a temp folder to sever the heavy computation graph
TEMP_PATH = "data/temp_top_100_subreddits"
# df_top_100.write.mode("overwrite").parquet(TEMP_PATH)

# df_ready = spark.read.parquet(TEMP_PATH)
df_ready = df_top_100

# Train-test split
train_data, test_data = df_ready.randomSplit([0.8, 0.2], seed=42)

# PIPELINE 
tokenizer = RegexTokenizer(inputCol="self_text", outputCol="words", pattern="\\W+")
remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
hashingTF = HashingTF(inputCol="filtered_words", outputCol="features", numFeatures=5000)
label_indexer = StringIndexer(inputCol=TARGET_COL, outputCol="label", handleInvalid="skip")

# Naive Bayes 
nb = NaiveBayes(featuresCol="features", labelCol="label", smoothing=0.1, modelType="multinomial")

pipeline = Pipeline(stages=[tokenizer, remover, hashingTF, label_indexer, nb])

# TRAIN & EVALUATE
try:
    print("Training Naive Bayes on the Top 100 subreddits... Please wait.")
    model = pipeline.fit(train_data)
    print("Model trained successfully!")
    
    print("Evaluating on test set...")
    predictions = model.transform(test_data)
    
    evaluator = MulticlassClassificationEvaluator(
        labelCol="label", 
        predictionCol="prediction", 
        metricName="accuracy"
    )
    
    accuracy = evaluator.evaluate(predictions)
    print(f"\n======================================")
    print(f"Top 100 Subreddits Test Accuracy: {accuracy * 100:.2f}%")
    print(f"======================================\n")

except Exception as e:
    print("\n--- Pipeline Failed ---")
    print(e)

Finding the top 100 most popular subreddits...
Writing intermediate data to disk to free up memory...
Training Naive Bayes on the Top 100 subreddits... Please wait.
Model trained successfully!
Evaluating on test set...

Top 100 Subreddits Test Accuracy: 61.26%



In [ ]:
# Save the entire fitted pipeline model

model_path = "models/5_19_26_MichaelModel_smoothing_low"
model.write().overwrite().save(model_path)

In [ ]:
train_predictions = model.transform(train_data)
evaluator.evaluate(train_predictions)

0.6143915204622935